In [1]:
%pip install langchain langchain-chroma langchain_groq


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content='Dogs are great companions, known for their loyalty and friendliness.',
        metadata = {'source': 'mammal-pets-doc'}
    ),
    Document(
        page_content= 'Cate are independent pets that often enjoy their own space.',
        metadata = {'source': 'mammal-pets-doc'}
    ),
    Document(
        page_content= 'Goldfish are popular pets for beginners, requiring relatively simple care.',
        metadata = {'source': 'fish-pets-doc'}
    ),
    Document(
        page_content= 'Parrots are intelligent birds capable of mimicking human speech',
        metadata = {'source': 'birds-pets-doc'}
    ),
    Document(
        page_content= 'Rabbits are social animals that need plenty of space to hop around.',
        metadata = {'source': 'mammal-pets-doc'}
    )
]

In [4]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cate are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'birds-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

In [21]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
groq_api_key = os.getenv('GROQ_API_KEY')

os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN')

llm = ChatGroq(groq_api_key = groq_api_key, model = 'llama-3.1-8b-instant')
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x13d308cd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x13d309310>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [7]:
%pip install langchain_huggingface


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name = 'all-MiniLM-L6-V2')

In [9]:
### VectoreStores
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents, embedding=embeddings)
vectorstore

In [10]:
vectorstore.similarity_search('cat')

[Document(id='1b978b19-5fd2-480c-8dd1-1242201de339', metadata={'source': 'mammal-pets-doc'}, page_content='Cate are independent pets that often enjoy their own space.'),
 Document(id='698854f4-57b2-4772-a447-48772042337b', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='818e5724-5db9-4f7e-9e2f-43943274aaa4', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='b0112502-dd76-4d50-9bae-87cafa705f72', metadata={'source': 'birds-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech')]

In [12]:
### Async Query
await vectorstore.asimilarity_search('cat')

[Document(id='1b978b19-5fd2-480c-8dd1-1242201de339', metadata={'source': 'mammal-pets-doc'}, page_content='Cate are independent pets that often enjoy their own space.'),
 Document(id='698854f4-57b2-4772-a447-48772042337b', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='818e5724-5db9-4f7e-9e2f-43943274aaa4', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='b0112502-dd76-4d50-9bae-87cafa705f72', metadata={'source': 'birds-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech')]

In [13]:
vectorstore.similarity_search_with_score('cat')

[(Document(id='1b978b19-5fd2-480c-8dd1-1242201de339', metadata={'source': 'mammal-pets-doc'}, page_content='Cate are independent pets that often enjoy their own space.'),
  0.9583950042724609),
 (Document(id='698854f4-57b2-4772-a447-48772042337b', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740900039672852),
 (Document(id='818e5724-5db9-4f7e-9e2f-43943274aaa4', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
  1.5956907272338867),
 (Document(id='b0112502-dd76-4d50-9bae-87cafa705f72', metadata={'source': 'birds-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech'),
  1.628485918045044)]

In [15]:
### Retrievers
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(['cat', 'dog'])

[[Document(id='1b978b19-5fd2-480c-8dd1-1242201de339', metadata={'source': 'mammal-pets-doc'}, page_content='Cate are independent pets that often enjoy their own space.')],
 [Document(id='698854f4-57b2-4772-a447-48772042337b', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [19]:
retriever = vectorstore.as_retriever(
    search_type = 'similarity',
    search_kwargs= {'k':1}
)

retriever.batch(['cat', 'dog', 'dianosaur'])

[[Document(id='1b978b19-5fd2-480c-8dd1-1242201de339', metadata={'source': 'mammal-pets-doc'}, page_content='Cate are independent pets that often enjoy their own space.')],
 [Document(id='698854f4-57b2-4772-a447-48772042337b', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')],
 [Document(id='b0112502-dd76-4d50-9bae-87cafa705f72', metadata={'source': 'birds-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech')]]

In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context: {context} 
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ('human', message)
    ]
)

rag_chain  = {'context': retriever, 'question': RunnablePassthrough()} | prompt | llm

responses = rag_chain.invoke('tell me about dogs')

print(responses.content)

Dogs are great companions, known for their loyalty and friendliness.
